# PyMolFit 04: Atmosphere and FITS metadata

Telluric transmission depends on the observation time, site, airmass, pressure, temperature, humidity, and vertical molecular profiles. This notebook shows how PyMolFit reads those quantities from a non-ESO Keck/HIRES spectrum, builds a MIPAS plus GDAS atmosphere, compares it with an explicit generic fallback, and measures the resulting change in the O2 correction.

PyMolFit does not silently invent coordinates for an unknown observatory. Missing geometry must be supplied explicitly or replaced by a documented fallback chosen by the scientist.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from pymolfit import AtmosphereProfile, correct_file, load_spectrum

candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next((p.resolve() for p in candidates if (p / "data").is_dir()), None)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Could not locate tutorials/data")
INPUT = TUTORIAL_ROOT / "data" / "hires_bd17_o2_topocentric_vacuum.fits"
OUTPUT = TUTORIAL_ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
FIT_RANGES = ((0.6868, 0.6905), (0.7592, 0.76515))

## Inspect the observing metadata

This compact FITS file retains the source observation time, Keck site geometry, airmass, and weather. The wavelength array is explicitly topocentric vacuum. Telluric lines belong to the observatory frame; PyMolFit can undo recognized barycentric or heliocentric products before fitting, but no frame conversion is required here.

In [ ]:
header = fits.getheader(INPUT, 0)
KEYS = (
    "OBJECT", "INSTRUME", "TELESCOP", "DATE-OBS", "AIRMASS", "SPECSYS",
    "ESO TEL GEOLAT", "ESO TEL GEOLON", "ESO TEL GEOELEV",
    "ESO TEL AMBI PRES START", "ESO TEL AMBI TEMP", "ESO TEL AMBI RHUM",
)
for key in KEYS:
    print(f"{key:28s}: {header.get(key)}")

spectrum = load_spectrum(INPUT, wavelength_medium="vacuum").to_unit("micron")
reference_wavenumber = float(np.nanmedian(1e4 / spectrum.wavelength))
print("Reference wavenumber [cm^-1]:", reference_wavenumber)

## Build the observation-specific atmosphere

`gdas_mode="auto"` first checks the verified local cache, then attempts the exact ESO GDAS profile bracketing the observation time and rounded site coordinates. If that profile cannot be obtained, it falls back to the packaged two-month average for the observation month. MIPAS supplies the upper atmosphere and additional molecular profiles. Inspect `gdas_source` instead of assuming that the network request succeeded.

In [ ]:
specific_atmosphere = AtmosphereProfile.from_fits_header_mipas_gdas(
    header,
    gdas_mode="auto",
    gdas_download_timeout_s=20.0,
    reference_wavenumber_cm=reference_wavenumber,
)

for key in (
    "observation_time_utc", "observatory_site", "observatory_coordinate_source",
    "observatory_altitude_m", "airmass", "gdas_source", "gdas_resolution",
):
    print(f"{key:32s}: {specific_atmosphere.metadata.get(key)}")

## Build an explicit generic fallback

When date or coordinates are unavailable, a scientist can build a layered standard atmosphere from known site conditions. This is less informative than time-local GDAS, but it is traceable and preferable to an undocumented hidden default. The compact HIRES header supplies the values below; for a metadata-poor spectrum they must come from the observing log or instrument documentation.

In [ ]:
generic_atmosphere = AtmosphereProfile.from_observatory_conditions(
    airmass=float(header["AIRMASS"]),
    observatory_altitude_m=float(header["ESO TEL GEOELEV"]),
    pressure_at_observatory_atm=float(header["ESO TEL AMBI PRES START"]) / 1013.25,
    temperature_at_observatory_k=float(header["ESO TEL AMBI TEMP"]) + 273.15,
)
print("Specific layers:", len(specific_atmosphere.layers))
print("Generic layers:", len(generic_atmosphere.layers))

## Compare the profiles

Pressure broadening and temperature-dependent line strengths vary by layer. Water vapour is especially time-variable; well-mixed O2 is less sensitive to weather, although pressure, temperature, and path geometry still affect its line shape.

In [ ]:
def profile_arrays(profile, default_site_altitude):
    vertical = np.array([
        layer.path_length_m if layer.vertical_path_length_m is None else layer.vertical_path_length_m
        for layer in profile.layers
    ])
    site = float(profile.metadata.get("observatory_altitude_m", default_site_altitude))
    edges = site + np.concatenate(([0.0], np.cumsum(vertical)))
    altitude_km = 0.5 * (edges[:-1] + edges[1:]) / 1000
    pressure = np.array([layer.pressure_atm for layer in profile.layers])
    temperature = np.array([layer.temperature_k for layer in profile.layers])
    h2o = np.array([layer.mixing_ratios.get("H2O", np.nan) for layer in profile.layers])
    return altitude_km, pressure, temperature, h2o

site_altitude = float(header["ESO TEL GEOELEV"])
specific_values = profile_arrays(specific_atmosphere, site_altitude)
generic_values = profile_arrays(generic_atmosphere, site_altitude)

figure, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
for values, label, color in ((specific_values, "MIPAS/GDAS", "tab:blue"), (generic_values, "Generic", "tab:orange")):
    altitude, pressure, temperature, h2o = values
    axes[0].semilogx(pressure, altitude, color=color, label=label)
    axes[1].plot(temperature, altitude, color=color)
    axes[2].semilogx(h2o, altitude, color=color)
axes[0].set_xlabel("Pressure [atm]")
axes[0].set_ylabel("Approximate altitude [km]")
axes[0].legend()
axes[1].set_xlabel("Temperature [K]")
axes[2].set_xlabel("H2O mixing ratio")
figure.suptitle("Observation-specific and generic atmosphere profiles")
figure.tight_layout()
plt.show()

for species in ("H2O", "O2"):
    specific_column = specific_atmosphere.total_column_cm2(species)
    generic_column = generic_atmosphere.total_column_cm2(species)
    print(f"{species}: specific={specific_column:.4e}, generic={generic_column:.4e}, ratio={specific_column / generic_column:.4f}")

## Fit the same O2 pixels with both atmospheres

Everything except the atmosphere is held fixed. O2 is selected explicitly because these predetermined windows are the O2 B and A bands. The purpose is sensitivity analysis, not choosing whichever atmosphere happens to resemble one correction better.

In [ ]:
def fit_with_atmosphere(atmosphere, label):
    started = time.perf_counter()
    fitted = correct_file(
        input_path=INPUT,
        wavelength_medium="vacuum",
        aer_catalog="auto",
        hitran_species=("O2",),
        atmosphere=atmosphere,
        fit_ranges=FIT_RANGES,
        continuum_order=3,
        solve_continuum_linear=True,
        lsf_sigma_pixels=0.9,
        fit_lsf_sigma=True,
        lsf_sigma_bounds=(0.3, 2.0),
        fit_wavelength_polynomial=True,
        wavelength_polynomial_order=1,
        wavelength_shift_bounds=(-5e-5, 5e-5),
    )
    print(f"{label}: {time.perf_counter() - started:.2f} s, scale={fitted.species_scales}, success={fitted.success}")
    return fitted

specific_result = fit_with_atmosphere(specific_atmosphere, "MIPAS/GDAS")
generic_result = fit_with_atmosphere(generic_atmosphere, "Generic")

In [ ]:
wave = specific_result.spectrum.to_unit("angstrom").wavelength
figure, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
for axis, (lower, upper), label in zip(axes, FIT_RANGES, ("O2 B band", "O2 A band"), strict=True):
    keep = (wave >= lower * 1e4) & (wave <= upper * 1e4)
    axis.plot(wave[keep], specific_result.transmission[keep], color="tab:blue", label="MIPAS/GDAS")
    axis.plot(wave[keep], generic_result.transmission[keep], color="tab:orange", linestyle="--", label="Generic")
    axis.set_ylabel("Transmission")
    axis.set_title(label)
    axis.legend()
axes[-1].set_xlabel("Vacuum wavelength [Angstrom]")
figure.tight_layout()
plt.show()

valid = np.isfinite(specific_result.transmission) & np.isfinite(generic_result.transmission)
difference = specific_result.transmission[valid] - generic_result.transmission[valid]
print("Transmission RMS difference:", np.sqrt(np.mean(difference**2)))
print("Maximum absolute difference:", np.max(np.abs(difference)))

## What to do when metadata is missing

Use this order of preference:

1. Recover the observation time, site, and weather from the original FITS header or observing log.
2. Supply `observatory_latitude_deg`, `observatory_longitude_deg`, `observatory_altitude_m`, and `airmass` explicitly to `correct_file`.
3. Supply a documented `AtmosphereProfile` or `atmosphere_table`.
4. Use `AtmosphereProfile.from_observatory_conditions` as an explicit generic fallback and include atmosphere sensitivity in the uncertainty budget.

`allow_default_observatory=True` opts into the legacy Paranal fallback. It should not be used merely to make an unknown-site error disappear.